# Small Wind Turbine for Modena  Part 2: blade design & optimisation

Part 1 answered *how much energy*. Part 2 answers the engineering questions:
**what does the blade look like, how good can it be, and what is the best design for Modena?**

We do the aerodynamics in code with **Blade Element Momentum (BEM)** theory  the same
method professional tools use. (If you want a visual cross-check later, you can rebuild this
blade in the free **QBlade** app on your own computer; it is optional, the physics here is real.)
Run **Runtime  Run all** and read down.


## 1. Airfoil and design choice
A blade section behaves like a little wing: it makes lift (useful) and drag (a loss). We use a
simple analytic airfoil and pick a design angle of attack where lift is high and drag is low.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
trapz = getattr(np, 'trapezoid', np.trapz)   # version-safe integration

alpha0 = np.radians(-4.0)          # zero-lift angle
Clmax, Cd0, kdrag = 1.3, 0.008, 0.02
def airfoil(alpha):
    Cl = np.clip(2*np.pi*(alpha - alpha0), -Clmax, Clmax)   # lift coefficient
    Cd = Cd0 + kdrag*Cl**2                                   # drag coefficient
    return Cl, Cd

B       = 3                  # number of blades
lam_d   = 7.0                # design tip-speed ratio (blade-tip speed / wind speed)
alpha_d = np.radians(6.0)    # design angle of attack
Cl_d    = float(airfoil(alpha_d)[0])
print(f'Design lift coefficient Cl = {Cl_d:.2f}  (L/D = {Cl_d/airfoil(alpha_d)[1]:.0f})')


## 2. The optimal blade shape
For maximum energy capture, theory (Betz optimum, including wake rotation) tells us the ideal
inflow angle at every point along the blade  and from it the ideal **chord** (width) and
**twist**. Notice the blade is wide and steeply twisted near the root and thin, nearly flat at
the tip: that is the signature shape of every modern wind-turbine blade.


In [ ]:
N = 40
x = np.linspace(0.10, 1.0, N)              # position along blade, r/R
lam_r = lam_d * x
phi_d = (2.0/3.0)*np.arctan(1.0/lam_r)     # optimum inflow angle
cR    = (8*np.pi*x)/(B*Cl_d)*(1-np.cos(phi_d))   # chord / R
twist = phi_d - alpha_d                    # built-in twist (radians)

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(x, cR, lw=2); ax[0].set_title('Blade planform (chord)')
ax[0].set_xlabel('position along blade  r/R'); ax[0].set_ylabel('chord / R'); ax[0].grid(alpha=.3)
ax[1].plot(x, np.degrees(twist), lw=2, color='C3'); ax[1].set_title('Blade twist')
ax[1].set_xlabel('position along blade  r/R'); ax[1].set_ylabel('twist [deg]'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'Twist: {np.degrees(twist[0]):.0f} deg at root  ->  {np.degrees(twist[-1]):.0f} deg at tip')


## 3. How good is this blade?  Cp vs tip-speed ratio
**BEM** splits the blade into rings, balances the aerodynamic forces against the momentum lost
by the air, and solves for the power at each operating speed. The result is the **power
coefficient Cp**  the fraction of the wind's energy captured. No turbine can beat the **Betz
limit of 0.593**; real drag and tip losses pull a good small blade down to ~0.40-0.45.


In [ ]:
def cp_at_lambda(lam):
    dr = x[1]-x[0]; Q = 0.0
    for i in range(N):
        xi, ci, th = x[i], cR[i], twist[i]
        lr = lam*xi; a, ap = 0.3, 0.01
        for _ in range(300):
            phi = np.arctan2((1-a), (1+ap)*lr)
            s, c_ = np.sin(phi), np.cos(phi)
            Cl, Cd = airfoil(phi - th)
            Cn = Cl*c_ + Cd*s; Ct = Cl*s - Cd*c_
            sigma = B*ci/(2*np.pi*xi)
            ftip = (B/2)*(1-xi)/(xi*s + 1e-9)
            F = max((2/np.pi)*np.arccos(np.clip(np.exp(-ftip),0,1)), 1e-4)
            a_new = 1.0/(4*F*s*s/(sigma*Cn+1e-9) + 1)
            den = 4*F*s*c_/(sigma*Ct+1e-9) - 1
            ap_new = (1.0/den) if abs(den) > 1e-6 else ap
            a_new = np.clip(a_new,0.0,0.5); ap_new = np.clip(ap_new,-0.2,1.0)
            if abs(a_new-a)<1e-6 and abs(ap_new-ap)<1e-6:
                a, ap = a_new, ap_new; break
            a, ap = 0.5*a+0.5*a_new, 0.5*ap+0.5*ap_new
        W2 = (1-a)**2 + (lr*(1+ap))**2
        Q += B*0.5*W2*Ct*ci*xi*dr
    return (lam*Q)/(0.5*np.pi)

lams = np.linspace(2, 12, 26)
cps  = np.array([cp_at_lambda(L) for L in lams])
Cp_peak = float(cps.max()); lam_peak = float(lams[cps.argmax()])

plt.figure(figsize=(7,4))
plt.plot(lams, cps, lw=2, marker='o', ms=3)
plt.axhline(0.593, ls='--', c='gray', label='Betz limit 0.593')
plt.scatter([lam_peak],[Cp_peak], c='red', zorder=5, label=f'peak Cp={Cp_peak:.3f} at lambda={lam_peak:.1f}')
plt.title('Aerodynamic performance of the designed blade')
plt.xlabel('tip-speed ratio  lambda'); plt.ylabel('power coefficient  Cp')
plt.ylim(0,0.6); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f'Peak Cp = {Cp_peak:.3f} at tip-speed ratio {lam_peak:.1f}')


## 4. Feed the real Cp back into the energy model
In Part 1 we assumed Cp = 0.35. Now we use the Cp our own blade actually achieves and see how
Modena's annual energy changes.


In [ ]:
from scipy.special import gamma
A10, k, alpha_sh = 2.25, 1.8, 0.25
rho, HOURS, HOME_KWH, CO2 = 1.225, 8760, 2700, 0.31
def weib(v,A,kk): v=np.asarray(v,float); return (kk/A)*(v/A)**(kk-1)*np.exp(-(v/A)**kk)

def aep_kwh(D, hub, Cp, v_in=3.0, v_rated=11.0, v_out=25.0):
    area = np.pi*(D/2)**2
    A_h  = A10*(hub/10.0)**alpha_sh
    vv   = np.linspace(0,30,1201)
    Prated = 0.5*rho*area*v_rated**3*Cp
    P = np.zeros_like(vv)
    m1=(vv>=v_in)&(vv<v_rated); P[m1]=0.5*rho*area*vv[m1]**3*Cp
    m2=(vv>=v_rated)&(vv<v_out); P[m2]=Prated
    return trapz(P*weib(vv,A_h,k),vv)*HOURS/1000.0

print(f'AEP at D=3 m, hub=18 m:')
print(f'   placeholder Cp 0.35 : {aep_kwh(3,18,0.35):.0f} kWh/yr')
print(f'   designed blade Cp   : {aep_kwh(3,18,Cp_peak):.0f} kWh/yr')


## 5. Optimisation: the best turbine Modena allows
Two design knobs left: rotor **diameter** and tower **height**. We compute annual energy across
a grid of both (using our designed Cp), then pick the best design that still fits sensible
residential limits (diameter <= 4 m, height <= 24 m). The white box is the allowed region; the
star is the winner.


In [ ]:
Ds = np.linspace(1,6,26); Hs = np.linspace(10,30,26)
GRID = np.array([[aep_kwh(D,H,Cp_peak) for D in Ds] for H in Hs])
feas = (Ds[None,:]<=4.0) & (Hs[:,None]<=24.0)
masked = np.where(feas, GRID, -1)
iH,iD = np.unravel_index(masked.argmax(), masked.shape)
bestD, bestH, bestAEP = Ds[iD], Hs[iH], GRID[iH,iD]

plt.figure(figsize=(7.5,5))
im = plt.imshow(GRID, origin='lower', aspect='auto',
                extent=[Ds.min(),Ds.max(),Hs.min(),Hs.max()], cmap='viridis')
plt.colorbar(im, label='Annual energy [kWh/yr]')
plt.gca().add_patch(plt.Rectangle((1,10), 4-1, 24-10, fill=False, ec='white', lw=2, ls='--'))
plt.scatter([bestD],[bestH], c='red', marker='*', s=220, edgecolor='white', zorder=5)
plt.title('Annual energy vs rotor diameter and tower height')
plt.xlabel('rotor diameter [m]'); plt.ylabel('tower / hub height [m]')
plt.show()

print(f'RECOMMENDED design for Modena (within limits):')
print(f'   rotor diameter : {bestD:.1f} m')
print(f'   tower height   : {bestH:.0f} m')
print(f'   annual energy  : {bestAEP:.0f} kWh/yr  ({bestAEP/HOME_KWH*100:.0f}% of an Italian home)')
print(f'   CO2 avoided    : {bestAEP*CO2:.0f} kg/yr')
print('   note: the optimum sits ON both limits -> Modena is resource-limited, not design-limited.')


## 6. The honest conclusion: wind vs solar in Modena
A good engineer compares alternatives. Modena has weak wind but strong sun (PVGIS irradiation
~1,765 kWh/m2/yr). Here is the best allowed turbine against a modest rooftop solar array.


In [ ]:
PV_YIELD = 1350   # kWh per kWp per year for Modena (PVGIS-based, conservative)
labels = ['Best wind\nturbine', 'Solar 1 kWp', 'Solar 3 kWp']
vals   = [bestAEP, PV_YIELD, 3*PV_YIELD]
plt.figure(figsize=(7,4.5))
bars = plt.bar(labels, vals, color=['#c0504d','#f0ad4e','#4f81bd'])
for b,v in zip(bars,vals):
    plt.text(b.get_x()+b.get_width()/2, v, f'{v:.0f}', ha='center', va='bottom')
plt.ylabel('Annual energy [kWh/yr]'); plt.title('Modena: wind vs rooftop solar')
plt.grid(axis='y', alpha=.3); plt.show()
print(f'A 3 kWp rooftop solar array produces about {3*PV_YIELD/bestAEP:.0f}x more energy than the best wind turbine here.')
print('Conclusion: in Modena, solar is the right clean-energy choice; wind only makes sense at windier sites.')


## What Part 2 gives you
You designed a wind-turbine blade from aerodynamic theory, computed its real efficiency with
BEM, found Modena's best possible turbine, and showed honestly that solar beats it here. That
full arc  build it, test it, and judge it fairly  is exactly the engineering judgement strong
programmes look for.

**Left for you to personalise:** swap in your real Global Wind Atlas A and k for Modena, verify
the solar number on pvgis.com, and write the conclusion in your own voice. The report draft
covers all of this.
